In [2]:
!pip install torch==2.7.1 tqdm==4.66.4 scikit-learn==1.4.2 pandas numpy

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 MB 5.9 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 6.2 MB/s  0:00:01m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 6.3 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 4.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 5.0 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Uninstalling tqdm-4.67.1:
      Successfully uninstalled tqdm-4.67.1
  Attempting uninstall: scikit-learn╸━━━━━━━━━━━ 5/7 [torch]kx]
    Found existing installation: scikit-learn 1.6.1━━━━━━━━━━━ 5/7 [torch]
    Uninstalling scikit-learn-1.6.1:╸━━━━━━━━━━━ 5/7 [torch]
      Successfully uninstalled scikit-learn-1.6.10m━━━━━━━━━━━ 5/7 [torch]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [scikit-learn] [scikit-learn]


In [86]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    LabelEncoder,
    OneHotEncoder,
    OrdinalEncoder,
)

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import accuracy_score, classification_report

In [95]:
df = pd.read_csv('data/other/stroke_data.csv')
df.sample(10)

,Unnamed: 0,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
4241,4241,17569,Male,41.0,0.0,0.0,Yes,Self-employed,Rural,93.52,31.5,Unknown,0
4254,4254,71900,Female,10.0,0.0,0.0,No,children,Urban,77.43,16.2,Unknown,0
2565,2565,68224,Male,54.0,0.0,0.0,Yes,Private,Rural,209.50,37.9,formerly smoked,0
3726,3726,32514,Male,1.8,0.0,0.0,No,children,Urban,68.80,NaN,Unknown,0
1472,1472,41271,Male,68.0,1.0,0.0,Yes,Govt_job,Urban,222.29,30.1,never smoked,0
5080,5080,22691,Female,29.0,0.0,0.0,Yes,Self-employed,Urban,90.52,28.0,NaN,0
310,310,45538,Female,43.0,0.0,0.0,Yes,Self-employed,Rural,115.22,21.2,NaN,0
1277,1277,33187,Female,6.0,0.0,0.0,No,children,Urban,201.25,NaN,Unknown,0
1532,1532,19735,Female,59.0,0.0,0.0,Yes,Private,Rural,79.18,52.8,formerly smoked,0
542,542,46438,Female,54.0,0.0,0.0,Yes,Self-employed,Urban,79.30,30.6,formerly smoked,0


In [96]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         5110 non-null   int64  
 1   id                 5110 non-null   int64  
 2   gender             5110 non-null   object 
 3   age                5110 non-null   float64
 4   hypertension       4380 non-null   float64
 5   heart_disease      4716 non-null   float64
 6   ever_married       5110 non-null   object 
 7   work_type          5110 non-null   object 
 8   Residence_type     5110 non-null   object 
 9   avg_glucose_level  5110 non-null   float64
 10  bmi                4909 non-null   float64
 11  smoking_status     4599 non-null   object 
 12  stroke             5110 non-null   int64  
dtypes: float64(5), int64(3), object(5)
memory usage: 519.1+ KB


In [97]:
df["stroke"].value_counts()

stroke
0    4861
1     249
Name: count, dtype: int64

In [98]:
df["stroke"].value_counts(normalize=True)

stroke
0    0.951272
1    0.048728
Name: proportion, dtype: float64

In [99]:
df.drop(columns=['Unnamed: 0', 'id'], inplace=True)
df.sample()

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
269,Female,65.0,1.0,0.0,Yes,Private,Rural,75.7,41.8,Unknown,0


In [100]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
num_cols

['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi', 'stroke']

In [101]:
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
cat_cols

['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

In [102]:
df.isna().sum()

gender                 0
age                    0
hypertension         730
heart_disease        394
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status       511
stroke                 0
dtype: int64

In [103]:
label_encoder = LabelEncoder()

In [104]:
df['gender'] = label_encoder.fit_transform(df['gender'])
df['ever_married'] = label_encoder.fit_transform(df['ever_married'])
df['work_type'] = label_encoder.fit_transform(df['work_type'])
df['Residence_type'] = label_encoder.fit_transform(df['Residence_type'])

In [105]:
df['smoking_status'] = label_encoder.fit_transform(df['smoking_status'])
label_encoder.classes_

array(['Unknown', 'formerly smoked', 'never smoked', 'smokes', nan],
      dtype=object)

In [106]:
df.loc[df['smoking_status'] == 4, 'smoking_status'] = np.nan

In [107]:
df.sample(3)

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
4746,0,60.0,NaN,0.0,1,0,0,111.79,23.6,3.0,0
3361,1,48.0,0.0,0.0,1,2,0,62.89,29.6,0.0,0
2703,0,26.0,0.0,0.0,0,2,1,100.31,38.6,2.0,0


In [61]:
median_imputer = SimpleImputer(strategy="median")

In [108]:
df[['bmi']] = median_imputer.fit_transform(df[['bmi']])

In [113]:
knn_imputer = KNNImputer(n_neighbors=31)

In [114]:
df['smoking_status'] = knn_imputer.fit_transform(df[['smoking_status']])
df['heart_disease'] = knn_imputer.fit_transform(df[['heart_disease']])
df['hypertension'] = knn_imputer.fit_transform(df[['hypertension']])

In [115]:
df.sample(3)

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
2602,0,69.0,0.000000,0.0,1,3,1,99.68,17.6,1.000000,0
4210,0,15.0,0.000000,0.0,0,4,1,91.16,38.0,1.381605,0
343,1,64.0,0.097945,0.0,1,0,1,239.64,34.6,1.000000,0


In [116]:
df.isna().sum()

gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64

In [117]:
data_stroke = df.loc[df['stroke'] == 1]
data_stroke.info()

<class 'pandas.core.frame.DataFrame'>
Index: 249 entries, 0 to 248
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             249 non-null    int64  
 1   age                249 non-null    float64
 2   hypertension       249 non-null    float64
 3   heart_disease      249 non-null    float64
 4   ever_married       249 non-null    int64  
 5   work_type          249 non-null    int64  
 6   Residence_type     249 non-null    int64  
 7   avg_glucose_level  249 non-null    float64
 8   bmi                249 non-null    float64
 9   smoking_status     249 non-null    float64
 10  stroke             249 non-null    int64  
dtypes: float64(6), int64(5)
memory usage: 23.3 KB


In [118]:
data_no_stroke = df.loc[df['stroke'] == 0]
data_no_stroke.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4861 entries, 249 to 5109
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             4861 non-null   int64  
 1   age                4861 non-null   float64
 2   hypertension       4861 non-null   float64
 3   heart_disease      4861 non-null   float64
 4   ever_married       4861 non-null   int64  
 5   work_type          4861 non-null   int64  
 6   Residence_type     4861 non-null   int64  
 7   avg_glucose_level  4861 non-null   float64
 8   bmi                4861 non-null   float64
 9   smoking_status     4861 non-null   float64
 10  stroke             4861 non-null   int64  
dtypes: float64(6), int64(5)
memory usage: 455.7 KB


In [119]:
data_no_stroke = data_no_stroke.sample(350)
data_no_stroke.info() 

<class 'pandas.core.frame.DataFrame'>
Index: 350 entries, 4289 to 1605
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             350 non-null    int64  
 1   age                350 non-null    float64
 2   hypertension       350 non-null    float64
 3   heart_disease      350 non-null    float64
 4   ever_married       350 non-null    int64  
 5   work_type          350 non-null    int64  
 6   Residence_type     350 non-null    int64  
 7   avg_glucose_level  350 non-null    float64
 8   bmi                350 non-null    float64
 9   smoking_status     350 non-null    float64
 10  stroke             350 non-null    int64  
dtypes: float64(6), int64(5)
memory usage: 32.8 KB


In [120]:
data = pd.concat([data_no_stroke, data_stroke])
data = data.sample(frac=1)
data

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
4602,1,30.0,0.000000,0.054283,0,2,1,106.03,26.7,0.0,0
113,0,45.0,0.000000,0.000000,1,2,0,224.10,56.6,2.0,1
68,1,59.0,0.000000,0.000000,1,2,1,86.23,30.0,1.0,1
5074,0,52.0,0.000000,0.000000,1,0,1,104.00,25.6,3.0,0
103,0,81.0,0.000000,1.000000,1,3,0,78.70,19.4,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...
2724,1,58.0,0.000000,0.000000,1,3,0,99.83,36.3,3.0,0
149,0,70.0,0.000000,1.000000,1,2,0,239.07,26.1,2.0,1
4137,0,28.0,0.097945,0.000000,1,2,0,105.90,28.6,3.0,0
1599,0,68.0,0.000000,0.054283,1,2,0,81.38,23.1,0.0,0


### Train/Validation Split and Scaling

We now:

1. Separate **X** (features) and **y** (target).
2. Split into training and validation sets using `train_test_split` (80% train, 20% validation). 
3. Use `StandardScaler` to scale features so that each numeric column has zero mean and unit variance.

Why scaling?
- Neural networks train more efficiently when inputs are on a similar scale.
- Gradient descent steps become more stable and less sensitive to the magnitude of features.


In [121]:
X = data.drop(columns=['stroke']).values.astype('float32') 
y = data['stroke'].values.astype('float32') 

X.shape, y.shape

((599, 10), (599,))

In [122]:
X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2)

X_train.shape, X_val.shape

((479, 10), (120, 10))

In [124]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

X_train[:3]

array([[-0.83942026, -0.36008146, -0.4457154 , -0.34469602,  0.57815385,
        -0.04970147, -1.0580496 , -0.7784829 , -0.13214445,  0.56708467],
       [-0.83942026, -0.04475344, -0.4457154 , -0.34469602,  0.57815385,
        -2.033619  ,  0.9451353 , -0.9834227 ,  1.7455026 ,  0.56708467],
       [ 1.1912984 , -0.22494087, -0.4457154 , -0.34469602,  0.57815385,
        -0.04970147,  0.9451353 ,  1.9354686 ,  3.549804  , -1.523789  ]],
      dtype=float32)

### PyTorch Dataset and DataLoader

To train neural networks efficiently, we use **mini-batches** instead of feeding all data at once.

Concepts:

- **Dataset**: an object that knows how to return a single sample (`(X[i], y[i])`).
- **DataLoader**: wraps a Dataset and:
  - creates batches,
  - shuffles the data for training,
  - can use multiple workers to load data in parallel.

We define a custom `StrokeDataset` that converts NumPy arrays into PyTorch tensors.


In [125]:
class StrokeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = StrokeDataset(X_train, y_train)
val_ds = StrokeDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

len(train_ds), len(val_ds)

(479, 120)

### Building a Neural Network in PyTorch

A **feed-forward neural network** (Multi-Layer Perceptron) is composed of layers of **neurons**:

- Each neuron computes:  
  \[ z = w \cdot x + b \]  
  where:
  - \( x \) is the input vector,
  - \( w \) is a vector of weights,
  - \( b \) is a bias term.


- Then we apply a non-linear **activation function**, such as **ReLU** (Rectified Linear Unit):  
  \[ \text{ReLU}(z) = \max(0, z) \]

Stacking multiple layers allows the network to learn complex, non-linear relationships between input features and the target.

### Our architecture

- Input layer: size = number of features
- Hidden layer 1: 64 neurons + ReLU
- Hidden layer 2: 32 neurons + ReLU
- Output layer: 1 neuron producing a **logit** (a raw score before applying sigmoid)

We will use loss which applies **sigmoid** internally to convert the logit to a probability in \([0, 1]\) and then computes **binary cross-entropy** loss.


In [126]:
class StrokeNet(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),  # one logit
        )

    def forward(self, x):
        # Output shape: (batch_size,)
        return self.net(x).squeeze(1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = StrokeNet(in_features=X_train.shape[1]).to(device)
model

Using device: cpu


StrokeNet(
  (net): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

### Loss Function and Optimizer

**Loss function**: measures how far the model's predictions are from the true labels.

We use:
- `nn.BCEWithLogitsLoss(pos_weight=...)`  
  - Combines a **sigmoid** activation and **binary cross-entropy** in a numerically stable way.
  - `pos_weight` tells the loss to **penalize errors on positive examples more**, to counter class imbalance.

**Optimizer**: algorithm that updates the model's weights to minimize the loss.

We use:
- `torch.optim.Adam`  
  - An adaptive gradient-based optimizer (a popular variant of stochastic gradient descent).
  - Takes learning rate `lr` as a key hyperparameter (we use `1e-3`).

In [127]:
pos_weight_tensor = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### Training loop steps for each epoch:

For each batch:
1. Move data to device (CPU or GPU).
2. Call `model(X_batch)` to get predictions (logits).
3. Compute loss by comparing predictions to true labels.
4. Call `loss.backward()` to compute gradients of loss w.r.t. each parameter.
5. Call `optimizer.step()` to update the parameters in the opposite direction of the gradient.
6. Call `optimizer.zero_grad()` before the next batch to reset gradients.

We repeat this for several **epochs** – full passes over the training data.

### Evaluation

To evaluate on the validation set:
1. Set `model.eval()` to switch off training-specific behavior (like dropout, if used).
2. Disable gradient computation with `torch.no_grad()` for efficiency.
3. Apply `torch.sigmoid` to logits to get probabilities.
4. Threshold at 0.5 to get class predictions (0 or 1).
5. Use `accuracy_score` and `classification_report` to summarize performance.


In [128]:
def train_one_epoch(epoch_idx: int):
    model.train()
    losses = []

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch_idx+1}", leave=False)

    for X_batch, y_batch in progress_bar:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    return float(np.mean(losses))


def evaluate():
    model.eval()
    all_probs = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)

            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)

            all_probs.append(probs)
            all_preds.append(preds)
            all_targets.append(y_batch.numpy())

    all_probs = np.concatenate(all_probs).reshape(-1)
    all_preds = np.concatenate(all_preds).reshape(-1)
    all_targets = np.concatenate(all_targets).reshape(-1)

    acc = accuracy_score(all_targets, all_preds)
    return acc, all_targets, all_preds

### Run Training

We now train the model for a fixed number of epochs (e.g., 20).

During training, we print:

- **Training loss**: should generally go down over time.
- **Validation accuracy**: helps us see how well the model generalizes to unseen data.

Note: Because of class imbalance, accuracy can be misleading; we will inspect precision/recall later.


In [129]:
EPOCHS = 20

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(epoch)
    val_acc, _, _ = evaluate()
    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"train_loss = {train_loss:.4f} | val_acc = {val_acc:.4f}"
    )

print("Training finished!")

Epoch 01/20 | train_loss = 5.8378 | val_acc = 0.5000


Epoch 02/20 | train_loss = 5.2262 | val_acc = 0.4250


Epoch 03/20 | train_loss = 4.4022 | val_acc = 0.4250


Epoch 04/20 | train_loss = 3.7951 | val_acc = 0.4250


Epoch 05/20 | train_loss = 3.1008 | val_acc = 0.4250


...

Epoch 16/20 | train_loss = 1.5942 | val_acc = 0.5750


Epoch 17/20 | train_loss = 1.6172 | val_acc = 0.5917


Epoch 18/20 | train_loss = 1.5741 | val_acc = 0.5833


Epoch 19/20 | train_loss = 1.5425 | val_acc = 0.5917


Epoch 20/20 | train_loss = 1.5938 | val_acc = 0.6000
Training finished!


In [130]:
val_acc, y_true, y_pred = evaluate()
print("Validation accuracy:", val_acc)
print()
print("Classification report:")
print(classification_report(y_true, y_pred, digits=3))

Validation accuracy: 0.6

Classification report:
              precision    recall  f1-score   support

         0.0      0.920     0.333     0.489        69
         1.0      0.516     0.961     0.671        51

    accuracy                          0.600       120
   macro avg      0.718     0.647     0.580       120
weighted avg      0.748     0.600     0.567       120

